# Módulo 3 · Notebook 3 — Base vectorial: Historia de los Mundiales

Indexamos el PDF **Historia de los mundiales (1930–2010)** en ChromaDB y lo guardamos de forma persistente en `data/vector_db_2/`.

**Requisitos:** Ollama en marcha con `nomic-embed-text:latest`.

## 1. Rutas y setup

In [6]:
import warnings
from pathlib import Path

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

warnings.filterwarnings("ignore")

BASE_DIR = Path("../../").resolve()
PDF_PATH = BASE_DIR / "data" / "Football" / "Historia de los Mundiales de Fútbol.pdf"
VECTOR_DIR = str(BASE_DIR / "data" / "vector_db_2")
COLLECTION_NAME = "mundiales_football"

Path(VECTOR_DIR).mkdir(parents=True, exist_ok=True)

print(f"📄 PDF    → {PDF_PATH}")
print(f"📁 Chroma → {VECTOR_DIR}")
print(f"   Existe PDF: {PDF_PATH.exists()}")

📄 PDF    → /Users/guane/Documentos/GlogalLogic/AI-course/data/Football/Historia de los Mundiales de Fútbol.pdf
📁 Chroma → /Users/guane/Documentos/GlogalLogic/AI-course/data/vector_db_2
   Existe PDF: True


## 2. Cargar el PDF

In [7]:
if not PDF_PATH.exists():
    raise FileNotFoundError(f"No se encontró el PDF: {PDF_PATH}")

loader = PyPDFLoader(str(PDF_PATH))
pages = loader.load()

print(f"📄 Páginas cargadas: {len(pages)}")
print(f"\n🔍 Vista previa (página 1):\n{pages[0].page_content[:400]}...")

📄 Páginas cargadas: 15

🔍 Vista previa (página 1):
Historia  de  los  Mundiales  de  Fútbol  
Un  análisis  exhaustivo  de  la  evolución  técnica,  social  y  global  del  torneo  de  la  FIFA  
 
Página  1  -  Introducción:  El  fenómeno  social  y  deportivo  global  
La  Copa  Mundial  de  la  FIFA  constituye,  sin  lugar  a  dudas,  el  acontecimiento  deportivo  y  cultural  
de
 
mayor
 
envergadura
 
y
 
resonancia
 
en
 
el
 
planeta
 
c...


## 3. Dividir en chunks

In [8]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
    length_function=len,
)

documents = text_splitter.split_documents(pages)

print(f"✂️  Chunks generados: {len(documents)}")
print(f"\n🔍 Ejemplo (chunk #5):\n{documents[5].page_content[:400]}...")

✂️  Chunks generados: 39

🔍 Ejemplo (chunk #5):
La  década  de  1930  atestiguó  la  rápida  consolidación  del  formato  competitivo  de  la  Copa  
Mundial,
 
pero
 
también
 
su
 
inevitable
 
instrumentalización
 
ideológica
 
en
 
un
 
contexto
 
europeo
 
marcado
 
por
 
el
 
ascenso
 
de
 
los
 
regímenes
 
totalitarios.
 
En
 
la
 
edición
 
de
 
Italia
 
1934,
 
el
 
régimen
 
fascista
 
de
 
Benito
 
Mussolini
 
utilizó
 
el
 
torneo
...


## 4. Crear y persistir ChromaDB

La primera ejecución puede tardar varios minutos (embedding de todos los chunks).

In [9]:
embeddings = OllamaEmbeddings(model="nomic-embed-text:latest")

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    persist_directory=VECTOR_DIR,
    collection_name=COLLECTION_NAME,
)

print(f"✅ Vector store guardado en: {VECTOR_DIR}")
print(f"   Colección: {COLLECTION_NAME}")
print(f"   Documentos indexados: {vectorstore._collection.count()}")

✅ Vector store guardado en: /Users/guane/Documentos/GlogalLogic/AI-course/data/vector_db_2
   Colección: mundiales_football
   Documentos indexados: 39


## 5. Cargar desde disco (sin re-indexar)

Si ya ejecutaste la celda anterior, usa esto en sesiones posteriores:

In [10]:
# embeddings = OllamaEmbeddings(model="nomic-embed-text:latest")
# vectorstore = Chroma(
#     persist_directory=VECTOR_DIR,
#     embedding_function=embeddings,
#     collection_name=COLLECTION_NAME,
# )
# print(f"✅ Cargado desde disco: {vectorstore._collection.count()} docs")

## 6. Prueba de búsqueda semántica

In [11]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

query = "¿Quién ganó el Mundial de 1978 y dónde se jugó?"
docs = retriever.invoke(query)

print(f"Pregunta: {query}\n")
for i, doc in enumerate(docs, 1):
    page = doc.metadata.get("page", "?")
    print(f"--- Resultado {i} (pág. {page + 1}) ---")
    print(doc.page_content[:350].strip())
    print()

Pregunta: ¿Quién ganó el Mundial de 1978 y dónde se jugó?

--- Resultado 1 (pág. 11) ---
La  gran  final,  disputada  en  el  Estadio  Internacional  de  Yokohama,  escenificó  un  choque  inédito  
pero
 
cargado
 
de
 
historia
 
entre
 
las
 
dos
 
naciones
 
más
 
laureadas
 
en
 
la
 
historia
 
de
 
la
 
competición:
 
Brasil
 
y
 
Alemania.
 
El
 
protagonismo
 
absoluto
 
de
 
la
 
velada
 
y
 
del
 
torneo
 
entero
 
le
 
corr

--- Resultado 2 (pág. 7) ---
tanto
 
en
 
la
 
presión
 
alta
 
asfixiante
 
como
 
en
 
la
 
gestación
 
lírica
 
del
 
ataque.
 
No
 
obstante,
 
a
 
pesar
 
de
 
su
 
innegable
 
superioridad
 
estética,
 
cayeron
 
en
 
la
 
gran
 
final
 
ante
 
el
 
pragmatismo
 
feroz
 
y
 
el
 
instinto
 
matador
 
de
 
la
 
Alemania
 
Occidental
 
de
 
Franz
 
Beckenbauer
 
y
 
Gerd

--- Resultado 3 (pág. 14) ---
triplete  histórico  de  Kylian  Mbappé  y  un  doblete  de  Messi—,  la  'Albiceleste'  se  impuso  en  una  
tanda
 
de
 
penales
 
cardíaca
 
graci